# Text-to-SQL with LangChain, Ollama, and Chinook

This cleaned notebook uses your local `qwen2.5:3b` model and the `Chinook.db` SQLite database. It removes the old Hugging Face/OpenAI setup and troubleshooting cells. Keep `Chinook.db` in the notebook's working directory and make sure Ollama is running with `qwen2.5:3b` pulled.

In [1]:
# Run once if these packages are missing
%pip install -U langchain langchain-community langchain-core langchain-ollama sqlalchemy

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 5.6 MB/s eta 0:00:01
   --------------------- ------------------ 1.3/2.4 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 5.1 MB/s eta 0:00:00
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.1.3
    Uninstalling SQLAlchemy-2.1.3:
      Successfully uninstalled SQLAlchemy-2.1.3
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
import re

C:\Users\Jafar tamboli\AppData\Local\Temp\ipykernel_8344\3011729487.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
# Connect to the database
# Make sure Chinook.db is in the notebook's current working directory.
db = SQLDatabase.from_uri("sqlite:///Chinook.db", sample_rows_in_table_info=0)

print("Database connected.")
print("Tables:", ", ".join(db.get_usable_table_names()))

Database connected.
Tables: Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track


In [4]:
# Initialize the local model
llm = ChatOllama(model="qwen2.5:3b", temperature=0)

In [5]:
# Build the SQL-generation chain
sql_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Text-to-SQL assistant. Generate one valid SQLite SELECT query using only the tables and columns in the supplied schema. Use exact table and column names. Return only SQL, with no explanation and no Markdown fences. Do not invent tables or columns."),
    ("human", "Database schema:\n{schema}\n\nQuestion: {question}\n\nSQL query:")
])

sql_chain = sql_prompt | llm | StrOutputParser()

def generate_sql(question):
    raw_sql = sql_chain.invoke({
        "schema": db.get_table_info(),
        "question": question
    }).strip()

    # Remove Markdown fences if the model includes them despite the instructions.
    raw_sql = re.sub(r"^```(?:sql)?\\s*|\\s*```$", "", raw_sql, flags=re.IGNORECASE).strip()
    return raw_sql

In [8]:
from langchain_ollama import ChatOllama

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0
)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a Text-to-SQL assistant.
Generate a valid SQLite query using only the provided database schema.
Use the exact table and column names from the schema.
Return only the SQL query. Do not include explanations or markdown fences."""),
    ("human", """Database schema:
{schema}

Question: {question}

SQL query:""")
])

chain = prompt | llm | StrOutputParser()

sql = chain.invoke({
    "schema": db.get_table_info(),
    "question": "Show me 2 random artists"
})

print(sql)

SELECT Name FROM Artist ORDER BY RANDOM() LIMIT 2


In [9]:
result = db.run(sql)
print(result)

[('Skank',), ('Big & Rich',)]


## Troubleshooting
- If `NameError` appears, run the cells from top to bottom.
- If SQLite reports `no such table`, check that `Chinook.db` is in the notebook's current working directory.
- If Ollama cannot connect, start Ollama and confirm `ollama list` includes `qwen2.5:3b`.
- A small model can still generate invalid SQL. Inspect the generated SQL before changing your database.